# S6E10: What Each Step Was Worth: Target Encoding, Teacher, 10 Folds, Stacking

**Problem.** Playground S6E10 asks for the probability that an airline passenger is satisfied, scored by ROC AUC. Dozens of public notebooks stack many models, but few say what each trick is actually worth.

**TL;DR**
- On one XGBoost with identical 5-fold CV, **in-fold target encoding (+0.00135)** and **a teacher trained on the original dataset (+0.00059)** were the two big steps, and both replicate in a live re-run on a subsample. Route-profile and auxiliary "expected rating" features added about +0.00014 each on full data, but in the live re-run the route profile was flat and the aux step was **too unstable to call**.
- **Training members on 10 folds instead of 5 improved every model we tried (+0.00009 to +0.00018)**, comparable to the aux step (+0.00014), for compute rather than a new idea.
- A plain logistic-regression stack beat every fancier stacker we tried. Submitted stack: **combiner CV 0.961794 → public LB 0.96134** (our submission of 2026-10-04). The CV→LB gap was similar (−0.00043/−0.00045) for both of our submissions.

| Result | CV | Public LB |
|---|---:|---:|
| **Submitted stack (9 members; sources at the end)** | 0.961794 | **0.96134** (historical: submitted 2026-10-04) |
| Secondary: stack of our own models only (7 members) | 0.961760 | not submitted |

**How this notebook runs.** The full models took about 5.5 GPU hours on Kaggle (plus several CPU hours for the LightGBM members). This notebook loads their saved out-of-fold and test predictions; only the 200,000-row feature ladder in section 2 is re-run live.

> **Terms.** *OOF (out-of-fold)*: each training row is predicted by a model that never saw it. *Combiner CV*: the stacker's weights are refit for every outer fold on the other folds' saved OOF predictions only. *Target encoding (TE)*: replace a value (e.g. an exact flight distance) by the satisfaction rate of training rows with that value, computed inside each fold. *Route profile*: for each exact `Flight Distance`, the mean of every other column over train+test (no labels). *Teacher*: a model trained only on the original 129,880-row airline dataset, whose prediction becomes a feature. *Aux features*: for each rating, its expected value predicted from the other 20 columns (no labels).

In [ ]:
import hashlib, json, os, platform, re, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb, xgboost as xgb, sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
T0 = time.time()
INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input'))

def find(name, hint):
    c = [p for p in INPUT.rglob(name) if hint in str(p).lower()]
    assert c, (name, hint)
    return sorted(c, key=lambda p: len(str(p)))[0]

TARGET, FD = 'satisfaction', 'Flight Distance'
RATINGS = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location',
           'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service',
           'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', FD, 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
FEATS = NUMS + RATINGS + CATS

train = pd.read_csv(find('train.csv', 'playground-series-s6e10'))
test = pd.read_csv(find('test.csv', 'playground-series-s6e10'))
sample = pd.read_csv(find('sample_submission.csv', 'playground-series-s6e10'))
orig = pd.read_csv(find('data.csv', 'aviation'))
y = train[TARGET].astype(str).str.lower().map({'true': 1, 'false': 0}).astype(int).to_numpy()
orig[TARGET] = orig[TARGET].astype(str).str.lower().map({'true': 1, 'false': 0, 'satisfied': 1,
                                                          'neutral or dissatisfied': 0}).astype(int)
orig = orig[FEATS + [TARGET]].drop_duplicates().reset_index(drop=True)
assert len(train) == 699_635 and len(test) == 299_844 and (sample['id'].values == test['id'].values).all()
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
fold_id = np.zeros(len(y), int)
for k, (_, b) in enumerate(FOLDS):
    fold_id[b] = k

# saved predictions: ours, plus two public CC0 OOF libraries on the same folds
OWN = find('members.json', 's6e10-stack-oof').parent
MEMBERS = json.loads((OWN / 'members.json').read_text())
mg_o = pd.read_parquet(find('oof_predictions.parquet', 's6e10-oof-library'))
mg_t = pd.read_parquet(find('test_predictions.parquet', 's6e10-oof-library'))
assert (mg_o['id'].values == train['id'].values).all() and (mg_t['id'].values == test['id'].values).all()
assert (mg_o['satisfaction'].astype(int).values == y).all() and (mg_o['fold'].values == fold_id).all()
GL = find('folds_seed42.npy', 'golem').parent
assert (np.load(GL / 'folds_seed42.npy') == fold_id).all()

def load(name):
    if name.startswith('pub_mg_'):
        p = name[7:]
        cols = [c for c in mg_o.columns if c == p or re.fullmatch(re.escape(p) + r'_s\d+', c)]
        return mg_o[cols].mean(axis=1).to_numpy(), mg_t[cols].mean(axis=1).to_numpy()
    if name.startswith('pub_gl_'):
        return np.load(GL / f'oof_{name[7:]}.npy'), np.load(GL / f'test_{name[7:]}.npy')
    f = OWN / f'{name}.npz'
    z = np.load(f if f.exists() else OWN / 'oof' / f'{name}.npz')
    return z['oof'], z['test']

auc = lambda p: roc_auc_score(y, p)
fold_aucs = lambda p: [roc_auc_score(y[b], p[b]) for _, b in FOLDS]
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
       'sklearn': sklearn.__version__, 'lightgbm': lgb.__version__, 'xgboost': xgb.__version__})
print(f'train {len(train):,} | test {len(test):,} | original {len(orig):,} | saved members {len(MEMBERS)}')

In [ ]:
def nested_stack(names, C=1.0):
    'Combiner CV: 5-fold logistic regression on saved member logits (weights refit per outer fold); returns pooled AUC, fold AUCs, OOF stack scores.'
    Z = np.column_stack([logit(load(n)[0]) for n in names])
    out = np.zeros(len(y))
    for a, b in FOLDS:
        out[b] = LogisticRegression(C=C, max_iter=3000).fit(Z[a], y[a]).decision_function(Z[b])
    return auc(out), fold_aucs(out), out

LADDER = [('raw 21 columns', 'xgb_raw'), ('+ target encoding (in-fold)', 'xgb_te'),
          ('+ route profile + counts', 'xgb_route'), ('+ original-data teacher', 'xgb_A'),
          ('+ aux expected ratings', 'xgb_A_auxev')]
FINAL = ['rmlp_v4aux_k10', 'pub_mg_lgb_ogteach', 'rmlp_raw_aux_k10', 'pub_gl_k', 'lgb_A_auxall_k10', 'pub_gl_g',
         'cat_ctr_auxev_k10', 'rmlp_ogx_aux', 'pub_mg_realmlp_v4']
OWN_ONLY = ["rmlp_v4aux_k10s2", "cat_ctr_auxall_k10", "rmlp_raw_aux_k10", "lgb_B_k10", "rmlp_ogx_aux_k10", "rmlp_v4ev_e4_k10", "lgb_A"]
ladder = pd.DataFrame([(lab, n, auc(load(n)[0])) for lab, n in LADDER], columns=['step', 'member', 'OOF AUC'])
best_single = max(['rmlp_v4aux_k10', 'rmlp_raw_aux_k10', 'lgb_A_auxall_k10', 'cat_ctr_auxev_k10'],
                  key=lambda n: auc(load(n)[0]))
own_auc = nested_stack(OWN_ONLY)[0]
final_auc, final_folds, _ = nested_stack(FINAL)
hero = pd.concat([ladder[['step', 'OOF AUC']], pd.DataFrame({
    'step': [f'best single model ({best_single}, 10-fold)', 'stack, our models only', 'stack + public OOF libraries'],
    'OOF AUC': [auc(load(best_single)[0]), own_auc, final_auc]})], ignore_index=True)
fig, ax = plt.subplots(figsize=(9, 4.6))
colors = ['#9ca3af'] + ['#2563eb'] * 4 + ['#059669'] * 3
ax.barh(hero['step'][::-1], hero['OOF AUC'][::-1] - 0.958, left=0.958, color=colors[::-1])
for i, v in enumerate(hero['OOF AUC'][::-1]):
    ax.text(v + 0.00003, i, f'{v:.5f}', va='center', fontsize=9)
ax.set_xlim(0.958, 0.9622)
ax.set_xlabel('5-fold out-of-fold ROC AUC (stacks: combiner CV on saved OOF)')
ax.set_title('Target encoding and the original-data teacher did most of the work;\nstacking added the last +0.0004', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
display(hero.round(6))
print(f'Submitted stack combiner CV {final_auc:.6f} -> public LB 0.96134 (submitted 2026-10-04) | our models only {own_auc:.6f} (not submitted)')

## 1. Validation design

One fold map for every model: `StratifiedKFold(5, shuffle=True, random_state=42)` over all 699,635 training rows. That is the split most public OOF libraries use, so their predictions line up row for row with ours (checked above: ids, labels and fold ids all match).

- **Target encodings** are fitted inside each training fold. Training rows get inner 5-fold encodings, so no row sees its own label.
- **Early stopping** uses a 5% inner split of each fold's training rows, never the validation fold. RealMLP trains a fixed number of epochs with no validation data.
- **Label-free features** (route profile, value counts, aux expected ratings, the original-data teacher) are computed once over train+test. They use no competition labels. This is transductive, standard in Playground, and stated here.
- **Stacks** are scored by *combiner CV on saved OOF predictions*: the logistic-regression weights are refit for each outer fold on the other four folds' OOF. Only the weights are nested. **Member selection (greedy, section 5) reused the same OOF**, so the stack CV is optimistic by an unknown amount.
- **Public leaderboard** feedback was never used to choose anything. We made two submissions, each after CV improved.

## 2. The feature ladder: one XGBoost, one change at a time

Same XGBoost (depth 8, learning rate 0.02), same folds; each row adds one block to the row above. These are full 5-fold runs from our GPU job, loaded from the saved predictions.

In [ ]:
lad = ladder.copy()
lad['gain vs previous'] = lad['OOF AUC'].diff()
lad['folds up vs previous'] = [None] + [sum(np.array(fold_aucs(load(LADDER[i][1])[0])) > np.array(fold_aucs(load(LADDER[i - 1][1])[0])))
                                        for i in range(1, len(LADDER))]
display(lad.round(6))

**Reading it.** The exact `Flight Distance` value behaves like a route ID: the synthetic generator keeps a satisfaction rate per value that a numeric split cannot isolate cheaply. **Target encoding** of distance and a few crosses captures it (+0.00135). The **original-data teacher** adds clean-label knowledge that 700k noisy synthetic rows do not recover on their own (+0.00059). The aux expected ratings improved all five folds; the route profile only three of five, so treat its +0.00014 as weak evidence.

### Live check on CPU (subsample)

To show the ladder is not an artefact of our saved files, the next cells rebuild the features from scratch and re-run the ladder with LightGBM on a **200,000-row stratified subsample, 3 folds**. Absolute AUCs are lower than full-data numbers (less data), but the direction of each step is what we check. These functions are written to be copied.

In [ ]:
def route_profile(full):
    'Per exact Flight Distance: mean of every other column, plus the route size. Label-free.'
    g = full.groupby(FD)
    out = pd.DataFrame({f'fdm_{c}': g[c].transform('mean').astype(np.float32) for c in FEATS if c != FD})
    out['fd_cnt'] = g['Age'].transform('size').astype(np.float32)
    return out

def value_counts(full, cols=NUMS):
    return pd.DataFrame({f'cnt_{c}': full[c].map(full[c].value_counts()).astype(np.float32) for c in cols})

def te_codes(full):
    'Integer keys whose satisfaction rate is target-encoded inside folds.'
    def key(cols):
        code = np.zeros(len(full), dtype=np.int64)
        for c in cols:
            k = pd.factorize(full[c], use_na_sentinel=False)[0]
            code = code * (k.max() + 1) + k
        return pd.factorize(code)[0]
    return {'te_fd': key([FD]), 'te_age': key(['Age']), 'te_fd_age': key([FD, 'Age']), 'te_fd_class': key([FD, 'Class']),
            'te_fd_travel': key([FD, 'Type of Travel']), 'te_seg': key(['Age', 'Class', 'Type of Travel', 'Customer Type'])}

def target_encode(k_fit, y_fit, k_apply, smooth=20.0):
    prior = y_fit.mean()
    s = pd.DataFrame({'k': k_fit, 'y': y_fit}).groupby('k')['y'].agg(['sum', 'count'])
    return pd.Series(k_apply).map((s['sum'] + prior * smooth) / (s['count'] + smooth)).fillna(prior).to_numpy(np.float32)

def add_te(keys, idx_fit, y_fit, idx_apply):
    'Fitting rows: inner 5-fold encodings. Apply rows: encodings from all fitting rows.'
    fit, app = {}, {}
    for name, codes in keys.items():
        col = np.zeros(len(idx_fit), np.float32)
        for i, j in KFold(5, shuffle=True, random_state=0).split(idx_fit):
            col[j] = target_encode(codes[idx_fit][i], y_fit[i], codes[idx_fit][j])
        fit[name], app[name] = col, target_encode(codes[idx_fit], y_fit, codes[idx_apply])
    return pd.DataFrame(fit), pd.DataFrame(app)

def original_teacher(orig, full):
    'XGBoost fitted on the original dataset only; its logit becomes a feature for competition rows.'
    m = xgb.XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', n_jobs=4, random_state=0)
    m.fit(orig[FEATS], orig[TARGET])
    return logit(m.predict_proba(full[FEATS])[:, 1]).astype(np.float32)

In [ ]:
t = time.time()
full = pd.concat([train[FEATS], test[FEATS]], ignore_index=True)
for c in CATS:
    cats = sorted(pd.concat([full[c], orig[c]]).unique())
    full[c] = pd.Categorical(full[c], categories=cats).codes.astype(np.int16)
    orig[c] = pd.Categorical(orig[c], categories=cats).codes.astype(np.int16)
RP, CNT, KEYS = route_profile(full), value_counts(full), te_codes(full)
TEACH = original_teacher(orig, full)
AUXEV = pd.read_parquet(OWN / 'aux_ev.parquet')
print(f'features built in {time.time() - t:.0f}s | teacher alone on competition labels: AUC {auc(TEACH[:len(y)]):.4f}')

sub_idx, _ = train_test_split(np.arange(len(y)), train_size=200_000, stratify=y, random_state=SEED)
sub_idx = np.sort(sub_idx)
LIVE_FOLDS = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(sub_idx, y[sub_idx]))
STEPS = [('raw 21 columns', []), ('+ target encoding', ['te']), ('+ route profile + counts', ['te', 'rp']),
         ('+ original-data teacher', ['te', 'rp', 'teacher']), ('+ aux expected ratings', ['te', 'rp', 'teacher', 'aux'])]

def frame(idx, blocks):
    X = full.iloc[idx][FEATS].reset_index(drop=True)
    parts = [X]
    if 'rp' in blocks: parts += [RP.iloc[idx].reset_index(drop=True), CNT.iloc[idx].reset_index(drop=True)]
    if 'teacher' in blocks: parts.append(pd.DataFrame({'teacher': TEACH[idx]}))
    if 'aux' in blocks: parts.append(AUXEV.iloc[idx].reset_index(drop=True))
    return pd.concat(parts, axis=1)

live = []
for name, blocks in STEPS:
    oof = np.zeros(len(sub_idx))
    for a, b in LIVE_FOLDS:
        ia, ib = sub_idx[a], sub_idx[b]
        Xa, Xb = frame(ia, blocks), frame(ib, blocks)
        if 'te' in blocks:
            ta, tb = add_te(KEYS, ia, y[ia], ib)
            Xa, Xb = pd.concat([Xa, ta], axis=1), pd.concat([Xb, tb], axis=1)
        i_fit, i_es = train_test_split(np.arange(len(ia)), test_size=0.05, stratify=y[ia], random_state=SEED)
        m = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8,
                               subsample_freq=1, colsample_bytree=0.5, n_jobs=4, verbose=-1, random_state=SEED)
        m.fit(Xa.iloc[i_fit], y[ia][i_fit], eval_set=[(Xa.iloc[i_es], y[ia][i_es])], eval_metric='auc',
              callbacks=[lgb.early_stopping(150, verbose=False)])
        oof[b] = m.predict_proba(Xb)[:, 1]
    live.append((name, roc_auc_score(y[sub_idx], oof), [roc_auc_score(y[sub_idx][b], oof[b]) for _, b in LIVE_FOLDS]))
live = pd.DataFrame(live, columns=['step', 'live OOF AUC (200k, 3 folds)', 'fold AUCs'])
live['gain vs previous'] = live.iloc[:, 1].diff()
live['fold AUCs'] = live['fold AUCs'].map(lambda v: ' '.join(f'{x:.5f}' for x in v))
display(live.round(6))
live.round(6).to_csv('live_ladder.csv', index=False)
print(f'live ladder took {time.time() - t:.0f}s')

**Reading it.** On 200k rows the two big steps replicate on every fold: target encoding (+0.0007) and the original-data teacher (+0.0007). The route profile is flat (+0.000006). The aux expected ratings are **unstable across runs**: +0.00013 in this run (two of three folds up), but −0.0002 (all three folds down) in a repeat run of the same notebook where the teacher came out very slightly different (AUC 0.9553 instead of 0.9551, from another XGBoost build). With only two runs we cannot say whether the step helps at this data size.

Our interpretation (a hypothesis): route-profile and aux features compress information a tree can eventually learn by itself, so their value depends on data size and on the rest of the stack. On full data they were small but positive, and all five members we contributed to the submitted stack use the aux features (section 5). If you have limited time, do target encoding and the teacher first.


## Why target encoding wins: Flight Distance as a lookup key

Target encoding was the biggest single step. Two quick checks on the training labels show why, and whether the generator left anything our models still miss.

In [ ]:
g = pd.DataFrame({'v': train[FD], 'y': y}).groupby('v')['y'].agg(['mean', 'size'])
g = g[g['size'] >= 200]
sampling_sd = np.sqrt((g['mean'] * (1 - g['mean']) / g['size']).mean())
adjacent_gap = np.abs(np.diff(g['mean'].to_numpy())).mean()
print(f'distances with >= 200 rows: {len(g)} | median rows per distance: {int(g["size"].median())}')
print(f'mean |satisfaction-rate gap| between neighbouring distances: {adjacent_gap:.4f}')
print(f'gap expected from sampling noise alone (SD):                {sampling_sd:.4f}  (ratio {adjacent_gap / sampling_sd:.1f}x)')

fig, ax = plt.subplots(figsize=(9, 3))
w = g.loc[(g.index >= 300) & (g.index <= 700)]
ax.plot(w.index, w['mean'], color='#2563eb', lw=1)
ax.plot(w.index, w['mean'].rolling(25, center=True).mean(), color='#9ca3af', lw=2, label='25-value rolling mean')
ax.set_xlabel('Flight Distance (300 to 700, values with >= 200 rows)'); ax.set_ylabel('satisfied rate')
ax.set_title('Neighbouring distances jump far more than sampling noise allows', loc='left')
ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

last = train[FD] % 10
rates = pd.DataFrame({'d': last, 'y': y}).groupby('d')['y'].agg(['mean', 'size'])
p_best = load('rmlp_v4aux_k10')[0]
res = pd.DataFrame({'d': last, 'r': y - p_best, 'v': p_best * (1 - p_best)}).groupby('d').agg(
    m=('r', 'mean'), n=('r', 'size'), v=('v', 'sum'))
digit = pd.DataFrame({'satisfied rate': rates['mean'], 'rows': rates['size'],
                      'model error z (rmlp_v4aux_k10)': res['m'] * res['n'] / np.sqrt(res['v'])})
digit.index.name = 'last digit of Flight Distance'
display(digit.round(4))
print(f'max |z| of model error across digits: {digit.iloc[:, 2].abs().max():.2f}')

**Measured.** Neighbouring distances differ in satisfied rate by far more than sampling noise would produce (printed above). The raw rate also depends on the last digit of the distance: distances ending in 0 are noticeably more often satisfied. But the out-of-fold errors of our best single model show no digit pattern (every |z| is below 2).

**Interpretation (consistent with the data, not proven).** The pattern is consistent with a generator that treats each exact distance as a key with its own satisfaction rate, rather than as a smooth quantity. That would explain why target-encoding the exact value beat everything else, and why a route profile adds little once it is there. The digit effect is already absorbed by models that see each exact value, so it is not a new signal to chase here.

## 3. Best single models

Four model families, each with the features above. RealMLP (pytabkit, public hyper-parameters, every numeric column also given as a categorical) is the strongest single model, and also gets the largest stack weight.

In [ ]:
fam = [('RealMLP + engineered inputs + aux', 'rmlp_v4aux_k10'), ('RealMLP, raw inputs + aux', 'rmlp_raw_aux_k10'),
       ('LightGBM + all features', 'lgb_A_auxall_k10'), ('CatBoost, numerics as categoricals', 'cat_ctr_auxev_k10'),
       ('XGBoost + all features (5-fold)', 'xgb_A_auxev')]
display(pd.DataFrame([(lab, n, MEMBERS[n]['nfold'], auc(load(n)[0])) for lab, n in fam],
                     columns=['model', 'member', 'training folds', 'OOF AUC']).round(6))

## 4. 10 folds beat 5 folds, for every model

Same model, same features, same seed; only the number of training folds changes (each fold model sees 90% instead of 80% of the rows). The OOF AUC is still pooled over all 699,635 rows, so the comparison is fair.

In [ ]:
pairs = [('rmlp_v4aux', 'rmlp_v4aux_k10'), ('rmlp_B', 'rmlp_B_k10'), ('rmlp_raw_aux_s2', 'rmlp_raw_aux_k10'),
         ('lgb_A_auxall', 'lgb_A_auxall_k10'), ('cat_ctr_auxev', 'cat_ctr_auxev_k10')]
kf = pd.DataFrame([(a, auc(load(a)[0]), auc(load(b)[0])) for a, b in pairs], columns=['model', '5-fold', '10-fold'])
kf['gain'] = kf['10-fold'] - kf['5-fold']
display(kf.round(6))
fig, ax = plt.subplots(figsize=(8, 2.8))
for i, r in kf.iterrows():
    ax.plot([r['5-fold'], r['10-fold']], [i, i], color='#9ca3af', zorder=1)
ax.scatter(kf['5-fold'], kf.index, color='#9ca3af', label='5 folds', zorder=2)
ax.scatter(kf['10-fold'], kf.index, color='#2563eb', label='10 folds', zorder=3)
ax.set_yticks(kf.index, kf['model']); ax.set_xlabel('OOF ROC AUC'); ax.legend(frameon=False, loc='lower right')
ax.set_title('Every model gained +0.00009 to +0.00018 from 10-fold training', loc='left')
ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

**Why it matters.** Ten-fold training gave +0.00009 to +0.00018 on every model, comparable to the aux step (+0.00014) and the route profile (+0.00014), without any new idea, at about 2× the training compute per model. It also helps the test predictions directly, because each fold model is trained on more data.

## 5. Stacking

Greedy forward selection over about 60 candidate members (ours plus two public CC0 OOF libraries). Each step is scored by combiner CV, stopping when a member adds less than 0.000005. The path below is recomputed live. Because the selection itself looked at the same OOF predictions, the later, smaller gains in this path are the least trustworthy.

In [ ]:
path = [(n, nested_stack(FINAL[:i + 1])[0]) for i, n in enumerate(FINAL)]
path = pd.DataFrame(path, columns=['added member', 'combiner CV AUC'])
path['gain'] = path['combiner CV AUC'].diff()
path['source'] = ['public library' if n.startswith('pub_') else 'ours' for n in path['added member']]
display(path.round(6))
Z = np.column_stack([logit(load(n)[0]) for n in FINAL])
w = LogisticRegression(C=1.0, max_iter=3000).fit(Z, y)
display(pd.DataFrame({'member': FINAL, 'weight': w.coef_[0]}).round(4))
print(f'our models only ({len(OWN_ONLY)} members): combiner CV {own_auc:.6f} | submitted, with public libraries: {final_auc:.6f}')

### A fancier stacker did not help

Same nine members, same outer folds.

In [ ]:
Zs = np.column_stack([logit(load(n)[0]) for n in FINAL])
res = {'mean of logits (no fitting)': auc(Zs.mean(1)), 'logistic regression': final_auc}
oof = np.zeros(len(y))
for a, b in FOLDS:
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=2000, subsample=0.8,
                           subsample_freq=1, colsample_bytree=0.8, reg_lambda=5, n_jobs=4, verbose=-1, random_state=0)
    oof[b] = m.fit(Zs[a], y[a]).predict_proba(Zs[b])[:, 1]
res['LightGBM stacker'] = auc(oof)
display(pd.Series(res, name='combiner CV AUC').round(6).to_frame())
pd.Series(res, name='combiner CV AUC').round(6).to_csv('stacker_compare.csv')

Our reading (a hypothesis, not tested): with nine highly correlated members there is little non-linear structure left for a tree stacker to find. We also tried a separate logistic regression per travel-type × class segment and segment interactions offline (combiner CV 0.961702 and 0.961700 against 0.961710 on an earlier 8-member set); neither beat the plain stack.

## 6. Does CV predict the leaderboard?

| Submission | Combiner CV | Public LB | Gap |
|---|---:|---:|---:|
| 8-member stack, 5-fold members | 0.961710 | 0.96128 | −0.00043 |
| 9-member stack, 10-fold members (this notebook) | 0.961794 | 0.96134 | −0.00045 |

Both submissions moved with CV by almost the same gap, and both were made only after CV improved. **Assumption:** the competition does not state its public/private split. If the public LB uses roughly 20% of the test rows (common in Playground), a single submission's public AUC carries noise of the order of ±0.0008. That would be larger than most gains in this notebook, which is why every decision here was made on CV.

## 7. What did not help (measured)

| Change | Result |
|---|---|
| A third round of 10-fold members and extra RealMLP seeds | stack +0.000015, only 3/5 folds up: noise |
| LightGBM stacker / per-segment LR / segment interactions | all below plain logistic regression (section 5) |
| LightGBM with numerics as native categoricals instead of target encoding | 0.960387 vs 0.961014 with TE (indicative: the categorical run also used learning rate 0.01 and another seed) |
| Expected ratings from original-only models on the "view B" LightGBM | 0.960794 vs 0.960863 without (different seeds) |
| Probability-of-actual-value aux features on top of expected values (XGBoost) | 0.960916 vs 0.961044 with expected values only (different seeds) |

## 8. Submission

The final stack is refit on all OOF rows and applied to the members' test predictions.

In [ ]:
T = np.column_stack([logit(load(n)[1]) for n in FINAL])
sub = sample[['id']].copy()
sub[TARGET] = 1 / (1 + np.exp(-w.decision_function(T)))
assert sub[TARGET].between(0, 1).all() and np.isfinite(sub[TARGET]).all() and len(sub) == 299_844
sub.to_csv('submission.csv', index=False)
print('submission.csv written | sha256', hashlib.sha256(open('submission.csv', 'rb').read()).hexdigest()[:16])
print(f'total runtime {(time.time() - T0) / 60:.1f} min')

## Takeaways

- **Do first:** target-encode the exact `Flight Distance` (and a few crosses) inside your folds, and add a teacher trained on the original dataset. Those two steps were worth about 0.002 AUC on one model.
- **Then:** train your best members on 10 folds; 10-fold gains were comparable to the smaller feature gains.
- **Stack simply:** logistic regression on logits, scored by combiner CV (and remember that selecting members on the same OOF flatters it). Diversity (RealMLP + trees + a second feature view) matters more than the stacker.
- **Trust CV:** our CV→LB gap was similar (about −0.0004) on both submissions.

**Limitations.**
- The full pipeline took about 5.5 GPU hours plus local CPU time; this notebook loads the saved predictions and re-runs only the 200k-row ladder live.
- Four of the nine submitted stack members are public OOF predictions (see below). No competition rule mentions this explicitly; we treat them as public external data. Our models alone reach 0.961760 combiner CV; that stack was not submitted.
- Stack CV is optimistic by an unknown amount because member selection reused the same OOF predictions.
- Label-free features use train+test together. One seed per model; seed variation of the full stack was not measured.
- **Teacher overlap:** 465 of the 129,880 original rows (0.36%) exactly or nearly match 474 competition rows (347 train, 127 test; 21 exact). Matching rule: all ratings and categories equal, Age within 1, Flight Distance within 10, delays within 5. Our teacher was trained on all original rows; none were excluded. No competition labels are involved, but the teacher's output on those few rows partly reflects the original label of a near-copy. The effect of excluding them was not measured here.
- **Early-stopping hygiene:** in the live ladder (and in our saved GBDT members), target encodings for training rows are built from all fitting rows, including the 5% held out for early stopping, so those rows' labels enter the encodings. Outer validation labels remained excluded, but the early-stopping procedure was not fully isolated; its effect on the reported AUCs has not been measured.

**Credits and sources.** Feature ideas (route profile, aux features, native-categorical CatBoost, original-data teacher): [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/s6e10-more-aux-carriers-lb-0-96152). RealMLP settings: [Vladimir Demidov](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit). 4 of the 9 stack members are public CC0 OOF predictions from [megayak](https://www.kaggle.com/datasets/megayak/s6e10-oof-library) and [dariushafshar](https://www.kaggle.com/datasets/dariushafshar/s6e10-golem-oof-library). Original data: [Binary Aviation Satisfaction 129k](https://www.kaggle.com/datasets/arseniyshutko/binary-aviation-satisfaction-129k) (CC0). Our saved predictions: [S6E10 Stack OOF Predictions](https://www.kaggle.com/datasets/sachith7/s6e10-stack-oof-predictions).